# AIE2026 第2回 発展課題 — 用語集LoRAと検索の比較

**目的**：講座の `data/glossary.json` を用いて SFT → LoRA 学習し、**学習前・学習後・用語集検索**を同じ質問で比較します。

- 公式 `scripts/make_sft_dataset.py` を実行して学習用 JSONL を作ります。
- 学習した用語、質問の言い換え、学習から除いた用語、用語集にない質問を分けて評価します。
- 出力文字列の類似度は **正答率ではありません**。回答の正しさはCSVの原文と定義を照合して判断してください。
- 公式の基本演習の結果（`eval_before.json` 等）は別に保存しておきます。

**実行方法**：Colab のGPUを T4 にして、上から順に実行。最後のセルで結果ZIPをダウンロードします。ノートブックの実行結果は本番GPU環境で確認する必要があります。

## 0. ライブラリの準備（公式ノートブックのバージョンと同じ）

In [ ]:
%pip uninstall -y -q torchao
%pip install -q "transformers==5.18.0" "peft==0.21.1" "trl==1.14.2" "accelerate==1.15.0" "datasets==5.1.0"
import torch, transformers, peft, trl, datasets
assert torch.cuda.is_available(), "GPUが見つかりません。ColabのランタイムをT4に変更してください。"
print("GPU:", torch.cuda.get_device_name(0))
print("versions:", {m.__name__:m.__version__ for m in (torch,transformers,peft,trl,datasets)})

## 1. 用語集データを生成、用語単位で学習と評価を分離

In [ ]:
import sys, os, json, random, subprocess, time, re, csv, shutil, difflib, statistics
from pathlib import Path
from collections import Counter, defaultdict
from datasets import Dataset

SEED=42
random.seed(SEED)
ROOT=Path('/content/aie_lec02_advanced'); ROOT.mkdir(parents=True,exist_ok=True)
REPO=ROOT/'source'
if not (REPO/'.git').exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/Akikukeo1/lecture-ai-engineering-2026.git',str(REPO)],check=True)
GLOSSARY=json.loads((REPO/'data/glossary.json').read_text(encoding='utf-8'))
TERMS=GLOSSARY['terms']
SFT_PATH=ROOT/'glossary_sft.jsonl'
with SFT_PATH.open('w',encoding='utf-8') as f:
    subprocess.run([sys.executable,str(REPO/'scripts/make_sft_dataset.py')],stdout=f,check=True)
records=[json.loads(line) for line in SFT_PATH.read_text(encoding='utf-8').splitlines()]
assert len(records)==2*len(TERMS)

# 用語そのものを holdout。既知用語の質問の言い換えとは異なる評価です。
rng=random.Random(SEED)
all_names=[t['term'] for t in TERMS]
shuffled=all_names.copy(); rng.shuffle(shuffled)
holdout=set(shuffled[:8]); known=[x for x in all_names if x not in holdout]
valid_terms=set(known[:10])
train_records=[]; valid_records=[]
for term in known:
    exact='「'+term+'」とは何ですか？'
    simple=term+'について簡単に説明してください。'
    hits=[r for r in records if r['messages'][0]['content'] in (exact,simple)]
    assert len(hits)==2, (term,len(hits))
    for r in hits:
        if term in valid_terms and r['messages'][0]['content']==simple: valid_records.append(r)
        else: train_records.append(r)
assert all(t['term'] not in holdout for t in TERMS if any('「'+t['term']+'」とは' in r['messages'][0]['content'] for r in train_records))
print(f"用語集 {len(TERMS)} 項目 / 生成 {len(records)} Q&A / train {len(train_records)} / valid {len(valid_records)} / 未学習用語 {len(holdout)}")
print('未学習用語:', sorted(holdout))
(ROOT/'splits.json').write_text(json.dumps({'seed':SEED,'known_terms':known,'heldout_terms':sorted(holdout),'n_train':len(train_records),'n_valid':len(valid_records)},ensure_ascii=False,indent=2),encoding='utf-8')

## 2. 評価質問と正解を作成

同じ質問セットを3方式に与えます。`既知（学習時と同形）` と `既知（言い換え）` は学習済みの用語に関する質問ですが、後者の**質問文自体は未学習**です。`用語自体を除外` は、用語の定義を学習データに含めません。用語集検索は完全な用語集にアクセスするため、この条件では検索側に情報アクセス上の優位があります。

In [ ]:
term_by_name={t['term']:t for t in TERMS}
known_test=known[:8]
heldout_test=sorted(holdout)[:8]
unknown_questions=[
 'AIE2026で定める「青葉プロトコル」とは何ですか？',
 'AIE2026の第99回の課題提出条件は何ですか？',
 'この講座の「六角状態分解法」を説明してください。',
 'AIE2026独自の「銀河キャッシュ規約」とは何ですか？',
 'この用語集の「赤紫シグナリング」の定義は？',
]
tests=[]
def add(category,term,question):
    tests.append({'id':len(tests)+1,'category':category,'term':term,'question':question,'reference':term_by_name[term]['definition'] if term else '用語集に記載なし'})
for term in known_test: add('学習済み用語',term,'「'+term+'」とは何ですか？')
for term in known_test: add('言い換え',term,term+'を一言で説明すると何ですか？')
for term in heldout_test: add('用語自体を除外',term,'「'+term+'」とは何ですか？')
for q in unknown_questions: add('用語集にない質問',None,q)
(ROOT/'eval_questions.json').write_text(json.dumps(tests,ensure_ascii=False,indent=2),encoding='utf-8')
print(Counter(t['category'] for t in tests))
print('質問例:',tests[:2],tests[-2:])

## 3. モデルを読み込み、LoRA学習前に評価

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
MODEL_NAME='Qwen/Qwen3.5-0.8B'
REVISION='2fc06364715b967f1860aea9cf38778875588b17'
tok=AutoTokenizer.from_pretrained(MODEL_NAME,revision=REVISION)
if tok.pad_token is None: tok.pad_token=tok.eos_token
model=AutoModelForCausalLM.from_pretrained(MODEL_NAME,revision=REVISION,dtype=torch.float32).to('cuda')
model.config.use_cache=True

def generate_answer(m,question):
    prompt=tok.apply_chat_template([{'role':'user','content':question}],tokenize=False,add_generation_prompt=True,enable_thinking=False)
    inputs=tok(prompt,return_tensors='pt').to(m.device)
    with torch.inference_mode():
        generated=m.generate(**inputs,max_new_tokens=190,do_sample=False,pad_token_id=tok.pad_token_id)
    return tok.decode(generated[0][inputs['input_ids'].shape[1]:],skip_special_tokens=True).strip()

def run_model(m,label):
    m.eval();rows=[]
    start=time.perf_counter()
    for i,t in enumerate(tests):
        t0=time.perf_counter();answer=generate_answer(m,t['question'])
        rows.append({**t,'method':label,'answer':answer,'elapsed_sec':round(time.perf_counter()-t0,2)})
        if (i+1)%8==0:print(label, i+1,'/',len(tests))
    print(label,'全評価時間:',round(time.perf_counter()-start,1),'秒')
    (ROOT/f'eval_{label}.json').write_text(json.dumps(rows,ensure_ascii=False,indent=2),encoding='utf-8')
    return rows

before_rows=run_model(model,'before')

## 4. 用語集Q&AでLoRA学習

学習の負荷を抑えるため、元の20エポックから**3エポック**に変更しています。学習前と学習後のモデル、評価質問、生成設定は同じです。少ないエポックで十分に学習できる保証はありません。

In [ ]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
train_ds=Dataset.from_list(train_records)
valid_ds=Dataset.from_list(valid_records)
model.config.use_cache=False
lora=LoraConfig(r=16,lora_alpha=32,lora_dropout=0.05,bias='none',task_type='CAUSAL_LM',target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])
settings=SFTConfig(output_dir=str(ROOT/'ckpt'),num_train_epochs=3,per_device_train_batch_size=4,gradient_accumulation_steps=2,learning_rate=2e-4,lr_scheduler_type='cosine',warmup_steps=3,logging_steps=5,save_strategy='no',fp16=False,max_length=512,packing=False,report_to='none',seed=SEED)
trainer=SFTTrainer(model=model,args=settings,train_dataset=train_ds,eval_dataset=valid_ds,peft_config=lora,processing_class=tok)
trainer.model.print_trainable_parameters()
t0=time.perf_counter()
training_output=trainer.train()
training_sec=round(time.perf_counter()-t0,1)
print('学習時間（秒）:',training_sec,'steps:',trainer.state.global_step,'final_loss:',training_output.training_loss)
trained=trainer.model
trained.config.use_cache=True
trained.eval()
(ROOT/'train_metadata.json').write_text(json.dumps({'base_model':MODEL_NAME,'revision':REVISION,'seed':SEED,'train_size':len(train_records),'valid_size':len(valid_records),'epochs':3,'lora_r':16,'lora_alpha':32,'learning_rate':2e-4,'training_seconds':training_sec,'global_steps':trainer.state.global_step,'training_loss':training_output.training_loss,'training_log':trainer.state.log_history},ensure_ascii=False,indent=2),encoding='utf-8')

## 5. LoRA学習後の評価

In [ ]:
after_rows=run_model(trained,'after')

## 6. 用語集検索（完全な58項目）との比較

これはLLMを使わない**完全一致に近い文字列照合**です。高度なRAGではありません。全用語集にアクセスするため、未学習用語でも検索によって定義を取得できます。

In [ ]:
def lookup(question):
    candidates=[]
    for t in TERMS:
        for name in [t['term'],*t.get('aliases',[])]:
            if len(name)>=2 and name.casefold() in question.casefold():
                candidates.append((len(name),t['definition'],t['term']))
    if not candidates:return '用語集に記載なし'
    candidates.sort(reverse=True)
    return candidates[0][1]
search_rows=[{**t,'method':'search','answer':lookup(t['question']),'elapsed_sec':0.0} for t in tests]
(ROOT/'eval_search.json').write_text(json.dumps(search_rows,ensure_ascii=False,indent=2),encoding='utf-8')

## 7. 比較・結果保存

ここで表示する**文字列類似度は正解率ではありません**。正誤は `evaluation_all.csv` の回答原文を参照し、用語集の定義との意味的一致を別途確認してください。未知質問の回答保留判定もキーワードによる暫定的なものです。

In [ ]:
import pandas as pd
from difflib import SequenceMatcher

def text_similarity(a,b):
    def clean(s):return re.sub(r'\s+','',s).casefold()
    return round(SequenceMatcher(None,clean(a),clean(b)).ratio(),3)
ABSTAIN=['確認が必要','わかりません','分かりません','不明','確認できません','情報がありません','記載がありません','記載なし','知りません']
all_rows=before_rows+after_rows+search_rows
for row in all_rows:
    row['reference_similarity']=(None if row['term'] is None else text_similarity(row['reference'],row['answer']))
    row['abstained_heuristic']=any(w in row['answer'] for w in ABSTAIN)
pd.DataFrame(all_rows).to_csv(ROOT/'evaluation_all.csv',index=False,encoding='utf-8-sig')
agg=[]
for method in ['before','after','search']:
    for cat in ['学習済み用語','言い換え','用語自体を除外','用語集にない質問']:
        rows=[r for r in all_rows if r['method']==method and r['category']==cat]
        agg.append({'method':method,'category':cat,'n':len(rows),
                    'mean_reference_similarity':None if cat=='用語集にない質問' else round(statistics.mean(r['reference_similarity'] for r in rows),3),
                    'abstention_rate_heuristic':round(sum(r['abstained_heuristic'] for r in rows)/len(rows),3)})
summary=pd.DataFrame(agg)
summary.to_csv(ROOT/'comparison.csv',index=False,encoding='utf-8-sig')
display(summary)
print('学習前後・検索の回答全文は evaluation_all.csv に保存しました。')
print('注意：平均文字列類似度は事実の正確性を測るものではありません。')

## 8. ファイルをダウンロード（必ず実行）

`result.json` と元の評価JSONは、基本演習用として別途保存しておいてください。第2回発展の結果は下記ZIPに集約します。

In [ ]:
from google.colab import files
ZIP_PATH=Path('/content/lec02_advanced_results.zip')
archive_files=['glossary_sft.jsonl','splits.json','eval_questions.json','eval_before.json','eval_after.json','eval_search.json','evaluation_all.csv','comparison.csv','train_metadata.json']
import zipfile
with zipfile.ZipFile(ZIP_PATH,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for name in archive_files:
        p=ROOT/name
        assert p.exists(),str(p)
        z.write(p,arcname=name)
print('保存:',ZIP_PATH)
files.download(str(ZIP_PATH))